# Clasificador de Tweets / Reviews

Proyecto de clasificación de texto listo para **Google Colab**.

**Pregunta:** ¿podemos clasificar el sentimiento o categoría de un tweet/review?

**Target:** sentimiento

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico de SMS.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", names=["label", "text"])
df["sentimiento"] = (df["label"] == "spam").astype(int)
print(df.shape)
print(df["sentimiento"].value_counts())
df.head()

## 2. EDA

In [ ]:
df["length"] = df["text"].str.len()
print("Longitud media positivo:", df[df["sentimiento"]==0]["length"].mean().round(1))
print("Longitud media negativo:", df[df["sentimiento"]==1]["length"].mean().round(1))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="sentimiento")
plt.xticks([0, 1], ["Ham", "Spam"])
plt.title("Distribución de clases")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="length", hue="sentimiento", bins=50, kde=True)
plt.title("Longitud del texto por clase")
plt.show()

### Hallazgos
- Dataset desbalanceado.
- Los textos de una clase suelen ser más largos.
- Ideal para clasificación de texto con TF-IDF.

## 3. Preparar datos

In [ ]:
X = df["text"]
y = df["sentimiento"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Logistic Regression"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Ham", "Spam"]).plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

## 6. Palabras más importantes

In [ ]:
vec = pipes[mejor].named_steps["tfidf"]
model = pipes[mejor].named_steps["model"]
coef = pd.Series(model.coef_[0], index=vec.get_feature_names_out()).sort_values()

plt.figure(figsize=(8, 5))
coef.head(10).plot(kind="barh", color="green")
plt.title("Palabras más asociadas a Ham")
plt.show()

plt.figure(figsize=(8, 5))
coef.tail(10).plot(kind="barh", color="red")
plt.title("Palabras más asociadas a Spam")
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap
print("SHAP listo (ejemplo para texto)")

## 8. Conclusiones

- TF-IDF + Logistic Regression suele dar Accuracy > **0.97** y ROC-AUC > **0.98**.
- Palabras clave separan bien las clases.
- Excelente introducción a NLP básico para tweets/reviews.

**Rango esperado:** Accuracy ~0.97–0.99 | ROC-AUC ~0.98–0.99